# Evidently Prompt Registry

### Let's run evidently ui service in background

In [ ]:
import evidently.ui.runner as service_runner

In [ ]:
runner = service_runner.EvidentlyUIRunner()

In [ ]:
runner.run()

In [ ]:
workspace = runner.get_workspace()

### Let's create a new project and load a prompt there

In [ ]:
project = workspace.create_project(name="Project with a prompt")
project_id = project.id

project_id

In [ ]:
prompt = workspace.prompts.get_or_create_prompt(project.id, "new prompt")
prompt.list_versions()

### Let’s add new versions of the prompt content. This helps you track changes over time.

In [ ]:
system_instruction = "This is a system instruction: initial version"
prompt.bump_version(system_instruction)

In [ ]:
prompt.list_versions()

In [ ]:
prompt.get_version().content

### You can now manage prompts: create, update, or delete versions

In [ ]:
prompt.bump_version("This is updated system instruction, say more detailed and more advanced")

In [ ]:
prompt.get_version("latest").content.as_text()

In [ ]:
prompt.delete_version(prompt.get_version().id)

In [ ]:
prompt.get_version("latest").content.as_text()

### You can also delete a prompt

In [ ]:
prompt_to_del = workspace.prompts.get_or_create_prompt(project.id, "prompt to delete")

In [ ]:
prompt_to_del.bump_version("some instructions")

In [ ]:
runner.show_service(service_runner.ServicePage(project_id=project_id, project_list='prompts'))

In [ ]:
prompt_to_del.delete()

In [ ]:
runner.show_service(service_runner.ServicePage(project_id=project_id, project_list='prompts'))

### Define a Judge with Criteria
Now, let’s define a **judge** that evaluates model responses using a template.
We’ll use a binary classification (GOOD / BAD) with simple criteria.


In [ ]:
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.descriptors import LLMJudge

judge = LLMJudge(provider="openai", model="gpt-4o-mini", template=BinaryClassificationPromptTemplate(
    target_category="GOOD",
    non_target_category="BAD",
    criteria="""Classify the model’s response with the following criteria:
Correctness: Is the response factually accurate?
Clarity: Is the response easy to understand?
Relevance: Does it fully address the question?
Output only one rating: good or bad."""
))

### Store the Judge Template in the Prompt Registry
Instead of keeping the template inline, let’s store it in the registry.

In [ ]:
template_prompt = workspace.prompts.get_or_create_prompt(project.id, "my template")
template_prompt.bump_version(judge.feature.template)

In [ ]:
template_prompt.list_versions()

### Reuse the Template
You can now load the template from the registry and create a new judge.

In [ ]:
new_judge = LLMJudge(provider="openai",
                 model="gpt-4o-mini",
                 template=template_prompt.get_version().content.template)
new_judge

### Terminate the service

In [ ]:
runner.terminate()